# W2-D1: Alert Correlation

M?c ti?u c?a notebook n?y l? gom nhi?u alert ri?ng l? th?nh ?t cluster h?n ?? b??c RCA ng?y sau x? l? d? h?n. Em d?ng 3 l?p ch?nh: fingerprint/dedup, session window theo th?i gian, v? topology-aware grouping d?a tr?n service graph.

## 1. Load dataset

Input g?m `lab/dataset/alerts_sample.jsonl` v? `lab/dataset/services.json`. Dataset m?u c? 20 alert, trong ?? scenario ch?nh l? payment/checkout/edge b? cascade, c?n recommender v? search l? 2 nh?m kh?c.

In [1]:
from pathlib import Path
import json

from correlate import load_jsonl, load_service_graph, correlate, semantic_similarity

BASE = Path.cwd()
alerts = load_jsonl(BASE / 'lab' / 'dataset' / 'alerts_sample.jsonl')
graph = load_service_graph(BASE / 'lab' / 'dataset' / 'services.json')

print('alerts:', len(alerts))
print('graph nodes:', len(graph))
print('first alert:', alerts[0]['id'], alerts[0]['service'], alerts[0]['metric'])

alerts: 20
graph nodes: 11
first alert: a-001 payment-svc db_pool_used_ratio


## 2. Run correlation

Em ch?n `gap_sec=120` v? ?? g?i ? 2 ph?t l? default h?p l?. N?u trong 120 gi?y kh?ng c? alert m?i th? xem nh? session c? k?t th?c. Em ch?n `max_hop=2` ?? gom c?c service g?n nhau trong graph nh?ng tr?nh gom qu? r?ng sang service kh?ng li?n quan.

In [2]:
result = correlate(alerts, graph, gap_sec=120, max_hop=2)
print(json.dumps({
    'input_alerts': result['input_alerts'],
    'output_clusters': result['output_clusters'],
    'reduction_ratio': result['reduction_ratio'],
    'parameters': result['parameters'],
}, indent=2))

{
  "input_alerts": 20,
  "output_clusters": 3,
  "reduction_ratio": 0.85,
  "parameters": {
    "gap_sec": 120,
    "max_hop": 2
  }
}


## 3. Inspect clusters

Cluster ch?nh c? payment-svc, checkout-svc v? edge-lb. Recommender c?ng th?i gian nh?ng kh?ng n?m tr?n c?ng service graph n?n b? t?ch ri?ng. Search x?y ra sau ?? ? session kh?c n?n c?ng th?nh cluster ri?ng.

In [3]:
for cluster in result['clusters']:
    print(
        f"{cluster['cluster_id']} | alerts={cluster['alert_count']} | "
        f"services={','.join(cluster['services'])} | "
        f"time={cluster['time_range'][0]} -> {cluster['time_range'][1]} | "
        f"max={cluster['max_severity']}"
    )

c-000-000 | alerts=14 | services=cart-svc,checkout-svc,edge-lb,notification-svc,payment-svc | time=2026-06-12T09:42:01Z -> 2026-06-12T09:48:30Z | max=crit
c-000-001 | alerts=2 | services=recommender-svc | time=2026-06-12T09:44:10Z -> 2026-06-12T09:45:20Z | max=crit
c-001-000 | alerts=4 | services=search-index,search-svc | time=2026-06-12T10:10:00Z -> 2026-06-12T10:11:55Z | max=warn


## 4. Write output

Output ch?nh ???c ghi v?o `results/cluster_summary.json` theo format c?a assignment.

In [4]:
out_path = BASE / 'results' / 'cluster_summary.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(out_path)
print('valid JSON:', json.loads(out_path.read_text(encoding='utf-8'))['output_clusters'])

D:\Study\Home_work\xBrain\aiops-dinhdanhnam-AIO1\w2\d1\results\cluster_summary.json
valid JSON: 3


## 5. Optional semantic similarity

Em l?m th?m layer semantic b?ng TF-IDF cosine tr?n text c?a fingerprint. Layer n?y gi?p t?m alert c? n?i dung gi?ng nhau, nh?ng kh?ng thay th? topology/time-window v? c?ng metric latency ? 2 service xa nhau v?n c? th? kh?ng li?n quan.


In [5]:
semantic = semantic_similarity(alerts)
semantic_path = BASE / 'results' / 'semantic_similarity.json'
semantic_path.write_text(json.dumps(semantic, ensure_ascii=False, indent=2), encoding='utf-8')
for pair in semantic['top_pairs'][:5]:
    print(f"{pair['left']} <-> {pair['right']} | similarity={pair['similarity']}")


notification-svc|queue_lag|warn <-> notification-svc|queue_lag|crit | similarity=0.8881
checkout-svc|http_5xx_rate|crit <-> payment-svc|http_5xx_rate|crit | similarity=0.7517
payment-svc|latency_p99_ms|crit <-> checkout-svc|latency_p99_ms|crit | similarity=0.7023
cart-svc|latency_p99_ms|warn <-> search-svc|latency_p99_ms|warn | similarity=0.6953
payment-svc|http_5xx_rate|crit <-> edge-lb|http_5xx_rate|crit | similarity=0.631


## 5. Short takeaway

Correlation kh?ng t?m root cause tr?c ti?p. N? gi?m alert flood tr??c: 20 alert ???c gom c?n 3 cluster, t?c reduction ratio = 0.85. Sau ?? RCA c? th? ch?y tr?n cluster ch?nh thay v? ??c t?ng alert ri?ng l?.